# Backup-кейсы для live demo

Короткий backup-notebook для защиты, если live FastAPI demo недоступен. Notebook не делает внешних API calls, не обучает модели и читает только готовые artifacts из `data/artifacts/reports`.

In [ ]:
from pathlib import Path
import csv
import json

ROOT = Path.cwd()
if not (ROOT / 'data').exists() and (ROOT.parent / 'data').exists():
    ROOT = ROOT.parent
REPORTS = ROOT / 'data' / 'artifacts' / 'reports'

def read_json(path):
    return json.loads(path.read_text(encoding='utf-8')) if path.exists() else {}

def read_csv(path, limit=None):
    if not path.exists():
        return []
    with path.open(encoding='utf-8', newline='') as file:
        rows = list(csv.DictReader(file))
    return rows[:limit] if limit else rows

def show_rows(rows, columns=None, limit=10):
    for row in rows[:limit]:
        if columns:
            print({column: row.get(column) for column in columns})
        else:
            print(row)


## 1. Статистика проекта

Показывает масштаб проекта: источники, candidate pairs, manual labels и ER metrics.

In [ ]:
readiness = read_json(REPORTS / 'ml_defense_readiness' / 'ml_defense_readiness_summary.json')
summary = read_json(REPORTS / 'ml_research_defense' / 'ml_research_defense_summary.json')
print('Readiness:', readiness)
print('Baseline keys:', list(summary.get('baseline_counts', {}).keys()))
print('ER metrics:', summary.get('existing_er_artifacts', {}).get('v3c_metrics', {}))


## 2. Demo-кейсы для защиты

Готовые кейсы: successful merge, rejected risky match и active learning candidates.

In [ ]:
demo_cases = read_csv(REPORTS / 'ml_research_defense' / 'defense_demo_cases.csv')
show_rows(demo_cases, ['case_type', 'item_a', 'item_b', 'score', 'interpretation'], limit=10)


## 3. Политика threshold

Показывает trade-off: высокий threshold повышает precision, но снижает recall.

In [ ]:
thresholds = read_csv(REPORTS / 'entity_resolution' / 'iterations' / 'manual_threshold_eval_v3c.csv')
show_rows(thresholds, ['threshold', 'predicted_positive', 'tp', 'fp', 'fn', 'precision', 'recall'])


## 4. Примеры рекомендаций

Content-based recommendations без user interactions.

In [ ]:
recommendations = read_csv(REPORTS / 'ml_research_defense' / 'recommendation_examples.csv')
show_rows(recommendations, ['seed_game', 'recommended_game', 'rank', 'score'], limit=10)


## 5. Grounded explanations

Русскоязычные explanations строятся из рассчитанных фактов. LLM не принимает решений.

In [ ]:
match_explanations = read_csv(REPORTS / 'rag_explanations' / 'match_explanation_examples.csv', limit=3)
recommendation_explanations = read_csv(REPORTS / 'rag_explanations' / 'recommendation_explanation_examples.csv', limit=3)
print('Match explanations:')
show_rows(match_explanations, ['subject', 'same_game_probability', 'model_decision', 'review_label'])
print('\nRecommendation explanations:')
show_rows(recommendation_explanations, ['seed_game', 'recommended_game', 'score'])


## 6. Bayesian rating

Дополнительный статистический блок: naive ratings корректируются количеством голосов.

In [ ]:
bayesian = read_json(REPORTS / 'bayesian_rating' / 'bayesian_rating_summary.json')
print(bayesian)
low_vote = read_csv(REPORTS / 'bayesian_rating' / 'low_vote_shrinkage_examples.csv', limit=5)
show_rows(low_vote)


## 7. Вывод для защиты

Главный тезис: проект демонстрирует не одну модель, а воспроизводимую data/ML-платформу, где ER-модель используется как управляемый scoring/manual-review слой, а рекомендации и explanations строятся поверх canonical facts.